# DCOPF Example 4 - PTDF-Based Formulation
by Xingpeng Li

Converted to Python/Pyomo by Haoxiang Wan

**Lab Website:** [rpglab.github.io](https://rpglab.github.io)

> **Note (e4 vs e1):** This example uses the **Power Transfer Distribution Factor (PTDF)** to express branch flows directly as linear functions of nodal injections. Voltage angle variables (`theta`) and line flow equations are eliminated. The PTDF-based line flow formula is:
> $$p_k = \sum_{n} \text{PTDF}_{k,n} \cdot (P_{G,n} - P_{D,n})$$
> where the slack/reference bus column is omitted (PTDF = 0 by convention).

In [ ]:
from pyomo.environ import *
import time

# ─────────────────────────────────────────────────────────────────
# System Data
# ─────────────────────────────────────────────────────────────────
BaseMW   = 100
refBus   = 3
total_load = 100  # MW at bus 2

buses = [1, 2, 3]

# Generators: {index: {bus, Pgmin, Pgmax, cost}}
gen_data = {
    1: {'bus': 1, 'Pgmin': 20, 'Pgmax': 70, 'c': 10},
    2: {'bus': 3, 'Pgmin': 40, 'Pgmax': 90, 'c': 20},
}

# Loads: {bus: MW}
load_data = {1: 0, 2: 100, 3: 0}

# Branches: {index: {from_bus, to_bus, rate MW}}
branch_data = {
    1: {'from': 1, 'to': 2, 'rate': 60},
    2: {'from': 1, 'to': 3, 'rate': 100},
    3: {'from': 2, 'to': 3, 'rate': 100},
}

# PTDF matrix: {(branch_k, bus_n): ptdf_value}
# Columns correspond to buses; reference bus (bus 3) has PTDF = 0
ptdf_data = {
    (1, 1):  0.25, (1, 2): -0.5,  (1, 3): 0.0,
    (2, 1):  0.75, (2, 2):  0.5,  (2, 3): 0.0,
    (3, 1):  0.25, (3, 2):  0.5,  (3, 3): 0.0,
}

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Build Pyomo Model
# ─────────────────────────────────────────────────────────────────
model = ConcreteModel()

# Sets
model.BUS    = Set(initialize=buses)
model.GEN    = Set(initialize=gen_data.keys())
model.BRANCH = Set(initialize=branch_data.keys())

# Parameters
model.busLoad       = Param(model.BUS,    initialize=load_data)
model.gen_bus       = Param(model.GEN,    initialize={g: gen_data[g]['bus']   for g in gen_data}, within=model.BUS)
model.gen_min       = Param(model.GEN,    initialize={g: gen_data[g]['Pgmin'] for g in gen_data})
model.gen_max       = Param(model.GEN,    initialize={g: gen_data[g]['Pgmax'] for g in gen_data})
model.gen_OpCost    = Param(model.GEN,    initialize={g: gen_data[g]['c']     for g in gen_data})
model.branch_rate   = Param(model.BRANCH, initialize={k: branch_data[k]['rate'] for k in branch_data})
model.PTDF          = Param(model.BRANCH, model.BUS, initialize=ptdf_data)

# Decision Variables (no theta needed in PTDF formulation)
model.G  = Var(model.GEN)
model.pk = Var(model.BRANCH)

# ── Objective ────────────────────────────────────────────────────
def obj_rule(model):
    return sum(model.gen_OpCost[g] * model.G[g] for g in model.GEN)
model.obj = Objective(rule=obj_rule, sense=minimize)

# ── Constraints ──────────────────────────────────────────────────
# Branch thermal limits
def branchLimits_rule(model, k):
    return inequality(-model.branch_rate[k], model.pk[k], model.branch_rate[k])
model.branchLimits = Constraint(model.BRANCH, rule=branchLimits_rule)

# PTDF-based line flow: pk = sum_n PTDF[k,n] * (Pg_n - Pd_n)
def lineFlowPTDF_rule(model, k):
    return model.pk[k] == sum(
        model.PTDF[k, n] * (
            sum(model.G[g] for g in model.GEN if model.gen_bus[g] == n)
            - model.busLoad[n]
        )
        for n in model.BUS
    )
model.lineFlowPTDF = Constraint(model.BRANCH, rule=lineFlowPTDF_rule)

# Generator capacity limits
def genLimits_rule(model, g):
    return inequality(model.gen_min[g], model.G[g], model.gen_max[g])
model.genLimits = Constraint(model.GEN, rule=genLimits_rule)

# Nodal power balance
def NodalPowerBalance_rule(model, n):
    gen_power = sum(model.G[g]  for g in model.GEN    if model.gen_bus[g]       == n)
    power_in  = sum(model.pk[k] for k in model.BRANCH if branch_data[k]['to']   == n)
    power_out = sum(model.pk[k] for k in model.BRANCH if branch_data[k]['from'] == n)
    return gen_power + power_in - power_out == model.busLoad[n]
model.NodalPowerBalance = Constraint(model.BUS, rule=NodalPowerBalance_rule)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Solve
# ─────────────────────────────────────────────────────────────────
solver = SolverFactory('gurobi')
solver.options['mipgap'] = 0.0
solver.options['timelimit'] = 90

start_time = time.time()
results = solver.solve(model, tee=True)
solve_time = time.time() - start_time

# ─────────────────────────────────────────────────────────────────
# Display Results
# ─────────────────────────────────────────────────────────────────
print('\n=== Generator Dispatch ===')
for g in model.GEN:
    print(f'  G[{g}] (Bus {gen_data[g]["bus"]}) = {model.G[g].value:.4f} MW')

print('\n=== Branch Flows (PTDF-based) ===')
for k in model.BRANCH:
    print(f'  pk[{k}] (Bus {branch_data[k]["from"]}→{branch_data[k]["to"]}) = {model.pk[k].value:.4f} MW')

total_cost = sum(gen_data[g]['c'] * model.G[g].value for g in model.GEN)
print(f'\n=== Total Cost = ${total_cost:.2f} ===')
print(f'Total solve elapsed time: {solve_time:.4f} seconds')